# The Probabilistic LSTM Forecaster
### Stage 2 of the Control Loop: Predicting Traffic as a Distribution, Not a Point

This notebook explains and demonstrates the forecaster that sits between the Digital Twin and the Live Agentic Planner. The full, tested implementation lives in `src/forecaster.py`.

> Why probabilistic, not a plain point forecast: see `notebooks/00_overview.ipynb`, Section 5. A point estimate hides exactly the information (uncertainty) the planner needs to make a risk-aware decision before a spike, not just a reactive one after it.


## 1. Model

A single-layer LSTM reads a sliding window of recent traffic and outputs two numbers: a predicted mean $\mu$ and a log-variance $\log(\sigma^2)$ for the next timestep. Predicting *log*-variance (not variance directly) means the network's raw output can be any real number — exponentiating it downstream always yields a valid, positive variance without needing a constrained output layer.

Trained by minimising Gaussian negative log-likelihood:

$$
\mathcal{L}_{\text{NLL}} = \frac{1}{2}\log\sigma^2 + \frac{(y - \mu)^2}{2\sigma^2}
$$

Minimising this jointly fits $\mu$ to the data **and** pushes $\sigma^2$ to reflect the model's actual squared error — a wrong prediction on a genuinely unpredictable point (e.g. right at the onset of a spike) is not punished as harshly as the same error would be on an easy point, *as long as the model reports high uncertainty there*.

*Implemented by:* `src/forecaster.py :: ProbabilisticLSTM`, `gaussian_nll_loss()`


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from digital_twin import DigitalTwin, SliceTrafficSpec
from forecaster import train_forecaster, predict, evaluate_calibration

# Reuse the same calibrated scenario from 02_digital_twin.ipynb, but run it
# for longer (500 steps) to have enough data to train on.
specs = [
    SliceTrafficSpec("URLLC", base_demand_mbps=22.0, noise_std_mbps=1.5,
                      spike_probability=0.03, spike_multiplier=1.9, spike_decay=0.65),
    SliceTrafficSpec("eMBB", base_demand_mbps=45.0, noise_std_mbps=6.0,
                      spike_probability=0.03, spike_multiplier=2.2, spike_decay=0.7),
]
twin = DigitalTwin(specs, fading_rho=0.9, fading_min_fraction=0.8, contention_strength=0.12, seed=7)
fixed_allocation = {"URLLC": 45.0, "eMBB": 60.0}

urllc_demand = []
for t in range(500):
    obs = twin.step(fixed_allocation)
    urllc_demand.append(obs["URLLC"].demand_mbps)

print(f"Generated {len(urllc_demand)} timesteps of URLLC demand from the real twin (src/digital_twin.py).")
print(f"mean={np.mean(urllc_demand):.2f}, std={np.std(urllc_demand):.2f}, max={max(urllc_demand):.2f}")


## 2. Training

Trained on the first 400 timesteps of URLLC demand from `02_digital_twin.ipynb`'s twin. The series is normalised (zero mean, unit std) internally before training, and predictions are mapped back to real Mbps automatically by `predict()`.

*Implemented by:* `src/forecaster.py :: train_forecaster()`


In [ ]:
train_series = urllc_demand[:400]

result = train_forecaster(train_series, window_len=10, hidden_size=24, epochs=250, lr=0.01, seed=42)

plt.figure(figsize=(7, 3.5))
plt.plot(result.train_losses, color='#1f4e8c')
plt.title('Training loss (Gaussian NLL) over 250 epochs')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"first-epoch loss: {result.train_losses[0]:.3f}   final loss: {result.train_losses[-1]:.3f}")


## 3. Rolling One-Step-Ahead Forecasts on Held-Out Data

The model has never seen timesteps 400–499. The plot below rolls the trained forecaster forward one step at a time over this held-out region, plotting the predicted mean with a shaded $\pm 1\sigma$ band against the actual demand.


In [ ]:
window_len = 10
preds_mean, preds_std, actuals = [], [], []

for i in range(400, 499):
    window = urllc_demand[i - window_len:i]
    p = predict(result, window)
    preds_mean.append(p["mean"])
    preds_std.append(p["std"])
    actuals.append(urllc_demand[i])

preds_mean = np.array(preds_mean); preds_std = np.array(preds_std); actuals = np.array(actuals)

plt.figure(figsize=(9, 4))
x = np.arange(len(actuals))
plt.plot(x, actuals, color='black', label='actual demand', linewidth=1.2)
plt.plot(x, preds_mean, color='#1f4e8c', label='predicted mean', linewidth=1.2)
plt.fill_between(x, preds_mean - preds_std, preds_mean + preds_std, color='#1f4e8c', alpha=0.2, label='±1σ')
plt.title('Rolling one-step-ahead forecast on held-out timesteps 400-499')
plt.xlabel('held-out timestep'); plt.ylabel('URLLC demand (Mbps)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((preds_mean - actuals) ** 2))
print(f"held-out RMSE: {rmse:.2f} Mbps")


## 4. Calibration — Does the Uncertainty Mean Anything?

Point-accuracy (RMSE above) only tells half the story. What the planner actually needs is a forecaster that is **honest** about its uncertainty: on held-out data, the fraction of true values landing inside the model's own $\pm1\sigma$ band should be in the right ballpark for a Gaussian (theoretically ~68% for $z=1$).

*Implemented by:* `src/forecaster.py :: evaluate_calibration()`


In [ ]:
test_series = urllc_demand[390:]  # includes 10 steps of history before the held-out region starts
coverage = evaluate_calibration(result, test_series, window_len=10, z=1.0)

print(f"1-sigma coverage on held-out data: {coverage:.1%}  (theoretical target for a well-calibrated Gaussian: ~68%)")
if coverage < 0.55:
    print("This run is somewhat overconfident (too-narrow intervals) -- expected from a small model trained")
    print("for a short time on ~400 points. More epochs, more data, or a slightly larger hidden size typically")
    print("closes this gap. The planner (Stage 4) should treat this as a known limitation, not ignore it.")


## 5. Why This Matters to the Next Stage

The Live Agentic Planner (`notebooks/01_agent_overview.ipynb` / `04_agentic_planner.ipynb`) is given `forecast_mean_mbps` and `forecast_std_mbps` per slice, straight from this model's `predict()` output, every control-loop timestep. A forecaster that is confidently wrong is more dangerous to the pipeline than one that is honestly uncertain — which is exactly why calibration (Section 4), not just RMSE, is tracked here as a first-class metric.


## 6. What's Implemented Where

| Concept | Function / Class | Tested by |
|---|---|---|
| Model architecture | `ProbabilisticLSTM` | forward-shape test |
| Gaussian NLL loss | `gaussian_nll_loss()` | confident-correct vs. confident-wrong ordering test |
| Sliding-window dataset | `make_windows()` | shape + too-short-series edge case |
| Training loop | `train_forecaster()` | loss-decreases-over-training smoke test |
| Inference | `predict()` | sane mean/positive std on a known synthetic series |
| Calibration metric | `evaluate_calibration()` | coverage lands in a broad sane range post-training |

All 7 tests pass — run with `PYTHONPATH=src pytest tests/test_forecaster.py -v`.

---
*Next: → `04_agentic_planner.ipynb`*
